# Exploratory Data Analysis — Credit Risk & Loan Default

Every table and figure below is produced by the `credit_risk` package; this notebook
loads the raw data, orchestrates the analysis, and records what it implies for modelling.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")


from credit_risk.data import RAW_CSV, query, read_frame
from credit_risk.eda import plots, profile

applicants = read_frame(RAW_CSV)
applicants.head()

## 1. Overview

In [ ]:
profile.overview(applicants)

In [ ]:
profile.numeric_summary(applicants)

In [ ]:
profile.categorical_summary(applicants)

## 2. Univariate distributions

In [ ]:
plots.plot_numeric_distributions(applicants)

In [ ]:
plots.plot_categorical_counts(applicants)

## 3. Missing values

The right-hand chart asks whether *missingness itself* predicts approval: a bar far from
the dashed base rate means imputing that column would erase signal.

In [ ]:
profile.missingness(applicants)

In [ ]:
plots.plot_missingness(applicants)

In [ ]:
mechanism = profile.missingness_mechanism(applicants)
mechanism

In [ ]:
plots.plot_missingness_mechanism(applicants)

In [ ]:
informative = mechanism.index[mechanism["mechanism"] == "informative"].tolist()
at_random = mechanism.index[mechanism["mechanism"] == "at random"].tolist()
print(f"Base approval rate: {mechanism.attrs['base_rate']:.1%}")
print(f"Informative missingness (imputation erases signal): {informative}")
print(f"Missing at random (safe to impute): {at_random}")

## 4. Outliers and impossible values

Statistical outliers (IQR, z-score) are separated from *impossible* values — negative
incomes and loan amounts that violate domain rules and are repaired before modelling.

In [ ]:
profile.outlier_summary(applicants)

In [ ]:
impossible = profile.outlier_summary(applicants)["impossible_negative"]
flagged = impossible[impossible > 0]
print("Impossible negative values to repair:")
print(flagged.to_string() if len(flagged) else "none")

## 5. Approval by feature

In [ ]:
plots.plot_boxplots_by_target(applicants)

In [ ]:
plots.plot_approval_rate_by_category(applicants)

In [ ]:
profile.approval_rate_by_band(applicants, "CreditScore")

In [ ]:
profile.approval_rate_by_band(applicants, "Income")

In [ ]:
query(
    "SELECT EmploymentType, ROUND(AVG(LoanApproved), 3) AS approval_rate, COUNT(*) AS n "
    "FROM applicants GROUP BY EmploymentType ORDER BY approval_rate DESC"
)

## 6. Association and correlation

Mutual information ranks every feature against the target; Pearson and Spearman agree when
a relationship is monotonic; the correlation ratio measures how much a numeric feature is
explained by a categorical one.

In [ ]:
profile.target_association(applicants)

In [ ]:
plots.plot_correlation_heatmaps(applicants)

In [ ]:
for column in ["CreditScore", "Income", "LoanAmount"]:
    eta = profile.correlation_ratio(applicants["EmploymentType"], applicants[column])
    print(f"correlation ratio  EmploymentType -> {column}: {eta:.3f}")

## 7. Multivariate structure

t-SNE standardizes mixed features; one-hot categoricals fragment the space into
combination islands, so read cluster *shapes* with care — separation here would be real
structure, its absence means the signal is marginal rather than a separable manifold.

In [ ]:
plots.plot_tsne(applicants)

In [ ]:
plots.plot_pca_scree(applicants)

## 8. Per-feature decision table

In [ ]:
profile.feature_decision_table(applicants)

## 9. Takeaways

In [ ]:
association = profile.target_association(applicants)
signal = association.index[association["signal"] == "signal"].tolist()
noise = association.index[association["signal"] == "noise"].tolist()
mechanism = profile.missingness_mechanism(applicants)
informative = mechanism.index[mechanism["mechanism"] == "informative"].tolist()

favorable = (
    (applicants["CreditScore"] > 580)
    & (applicants["EmploymentType"] != "Unemployed")
    & (applicants["Income"] > 40_000)
)
base_rate = applicants["LoanApproved"].mean()

print(f"Signal features ({len(signal)}): {signal}")
print(f"Noise features ({len(noise)}): {noise}")
print(f"Informative missingness: {informative}")
print(
    f"Jointly favorable applicants are {favorable.mean():.0%} of rows and approve at "
    f"{applicants.loc[favorable, 'LoanApproved'].mean():.0%} versus a {base_rate:.0%} base rate"
)

- **Signal is concentrated.** A few features carry the approval signal; the rest are noise
  and are dropped — `Gender` and `City` also on fair-lending grounds, and kept only for bias
  auditing.
- **Missingness can carry signal.** Where the present/missing bars diverge, imputation would
  destroy information for linear models — keep an indicator or let a tree split on the gap.
- **Approval is a soft AND.** The drivers must hold together, so engineered interactions are
  what will move a nonlinear model.